In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import altair as alt

# **API Load of GeoJson Files**

Takes in files from [Sound IQ ](https://experience.arcgis.com/experience/5416ae7c51ae4244a1a42ba679c1479f/page/KELP) filtering them for the beds we care about.


Assings them to the kelp geo data frame (kelpgdf)

In [ ]:
url = ('https://services.arcgis.com/6lCKYNJLvwTXqrmp/arcgis/rest/services/SIQ_Kelp_Surveys/FeatureServer/24/query?where=1%3D1&outFields=*&outSR=4326&f=geojson')
kelp_beds = gpd.read_file(url)

sites = ['Possession Point', 'Lowell Point', 'Hoypus Point', 'Polnell Point', "Ebey's Landing"]

kelpgdf = kelp_beds[kelp_beds['Location'].isin(sites)]
kelpgdf.head()

In [ ]:
kelpgdf.info()

In [ ]:
kelpgdf.value_counts('Location')

## **SoundIQ Data Cleaning**

We surely dont need this many columns and we dont want temperatures for all 4 data points so we can collapse those. (We may want to do something similar for depth? But depth isnt quite as complicated so.)

In [ ]:
kelp_red = kelpgdf[[
    'Location',
    'site_code',
    'Survey_Date',
    'Survey_Day',
    'Survey_Month',
    'Survey_Year',
    'Acres_NWSC_clip',
    'Miles',
    'Temp',
    'Temp1_Shore_Edge',
    'Temp1_Water_Edge',
    'Temp2_Shore_Edge',
    'Temp2_Water_Edge',
    'Ave_Temp_Shore_Edge',
    'Ave_Temp_Water_Edge',
    'Depth1_Shore_Edge',
    'Depth1_Water_Edge',
    'Depth2_Shore_Edge',
    'Depth2_Water_Edge']]

kelp_red.info()

### Temperature Averaging from Data Foundations

In [ ]:
def prepare_temp(df):

    temp_cols = [
        "Temp",
        "Ave_Temp_Shore_Edge",
        "Ave_Temp_Water_Edge",
        "Temp1_Shore_Edge",
        "Temp2_Shore_Edge",
        "Temp1_Water_Edge",
        "Temp2_Water_Edge",
    ]

    for c in temp_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")


    df["Temp_Final"] = np.where(
        ~df["Temp"].isna(),
        df["Temp"],
        np.where(
            (~df["Ave_Temp_Shore_Edge"].isna()) &
            (~df["Ave_Temp_Water_Edge"].isna()),
            (df["Ave_Temp_Shore_Edge"] + df["Ave_Temp_Water_Edge"]) / 2,
            (
                ((df["Temp1_Shore_Edge"] + df["Temp2_Shore_Edge"]) / 2) +
                ((df["Temp1_Water_Edge"] + df["Temp2_Water_Edge"]) / 2)
            ) / 2
        )
    )

    return df.head()

prepare_temp(kelp_red)

### Slight Renaming for Clarity

In [ ]:
kelp_clean= kelp_red[[
    'Location',
    'site_code',
    'Survey_Date',
    'Survey_Day',
    'Survey_Month',
    'Survey_Year',
    'Acres_NWSC_clip',
    'Miles',
    'Temp_Final',
    'Depth1_Shore_Edge',
    'Depth1_Water_Edge',
    'Depth2_Shore_Edge',
    'Depth2_Water_Edge']].rename(columns={
         'Acres_NWSC_clip' : 'area_acres',
         'Miles': 'perimeter_miles',
         'Temp_Final' : 'sstemp'
    })

kelp_clean['Location'] = kelp_clean['Location'].replace({
    "Ebey's Landing": 'Ebeys Landing'})

kelp_clean.info()

In [ ]:
kelp_clean['Location'].unique()

### Date Time Conversion

In [ ]:
kelp_clean['Survey_Date'] = pd.to_datetime(kelp_clean['Survey_Date'])
kelp_clean.info()

# **Locally Hosted Data Sets**

lat lon coordinates for all beds in MRC.

Conductivity, Temperature, & Dissolved Solid (CTD - Salinity/Temp) measurements from 5 beds in 2025 (Possession, Hoypus, Lowell, Ebeys (August only), Polnell (August Only).

In [ ]:
locations = pd.read_csv("/content/latlon_sites.csv")
ctd_2025 = pd.read_csv("/content/kelp_bed_ctd_2025.csv")

## **Locations Cleaning**

All that needs to be done is dropping the ones we dont have the other data for, specifically ctd.

In [ ]:
locations.head()

In [ ]:
locations.info()

In [ ]:
loc_sites = ['Possession Point', 'Lowell Point Camano State Park', 'Hoypus', 'Polnell', "Ebey's"]

clean_loc = locations[locations['Location'].isin(loc_sites)]

In [ ]:
clean_loc['Location'] = clean_loc['Location'].replace({
    'Lowell Point Camano State Park': 'Lowell Point',
    'Hoypus': 'Hoypus Point',
    'Polnell': 'Pollnell Point',
    "Ebey's": 'Ebeys Landing'})

In [ ]:
clean_loc.head()

In [ ]:
clean_loc.info()

## **CTD Cleaning**

Not any cleaning neccesary (I think), was built to not need cleaning.

In [ ]:
ctd_2025.head()

In [ ]:
ctd_2025.info()

In [ ]:
ctd_2025['site'].unique()

## Locations Mapped -- Ahrial

In [ ]:
import folium
import requests
import altair as alt

# king county
item_id = "ff8f49238a7040ac9efcb65bf708b24d"
meta_url = f"https://kingcounty.maps.arcgis.com/sharing/rest/content/items/{item_id}?f=json"
service_url = requests.get(meta_url).json()["url"]

# counties
query_url = f"{service_url}/0/query?where=1%3D1&outFields=*&outSR=4326&f=geojson"
geojson_data = requests.get(query_url).json()

# initlize on washington
kelpbed_map = folium.Map(
    location=[47.50, -120.74],
    zoom_start=7,
    tiles="https://arcgisonline.com{z}/{y}/{x}",
    attr="Esri",
    width=700,
    height=480
)

# counties
folium.GeoJson(
    geojson_data,
    style_function=lambda feature: {
        "fillColor": "#e8ecef",
        "color": "#4a5568",
        "weight": 0.8,
        "fillOpacity": 0.5,
    }
).add_to(kelpbed_map)

# kelp beds
for _, row in locations.iterrows():
    tooltip_text = (
        f"<b>Location:</b> {row['Location']}<br>"
        f"<b>Latitude:</b> {row['lat']:.4f}<br>"
        f"<b>Longitude:</b> {row['lon']:.4f}"
    )
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6,
        color="darkgreen",
        fill=True,
        fill_color="darkgreen",
        fill_opacity=0.85,
        tooltip=folium.Tooltip(tooltip_text)
    ).add_to(kelpbed_map)

# map
kelpbed_map


## Just Ebey's landing

In [ ]:
ebey_loc = locations[locations['Location'] == "Ebey's"]
ebey_loc

In [ ]:
target_lon = float(ebey_loc['lon'].iloc[0])
target_lat = float(ebey_loc['lat'].iloc[0])

background = alt.Chart(alt.Data(values=geojson_data["features"])).mark_geoshape(
    fill='#e8ecef',
    stroke='#4a5568',
    strokeWidth=0.8
)

points = alt.Chart(ebey_loc).mark_circle(
    color='#2ca02c',
    stroke='#1b661b',
    strokeWidth=1.5,
    size=100,
    opacity=1.0
).encode(
    longitude='lon:Q',
    latitude='lat:Q',
    tooltip=[
        alt.Tooltip('Location:N', title='Location'),
        alt.Tooltip('lat:Q', title='Latitude', format='.4f'),
        alt.Tooltip('lon:Q', title='Longitude', format='.4f')
    ]
)


ebey_map = (background + points).project(
    type='mercator',
    center=[target_lon, target_lat],
    scale=25000
).properties(
    width=700,
    height=480
)

ebey_map


## Map (lat-lon + ctd)

## Trend Analysis of kelp and regression

In [ ]:
idx = kelp_clean.groupby(['Location','Survey_Year'])['area_acres'].idxmax()
# keep the entire row
max_acres = kelp_clean.loc[idx]

max_acres

## Use the geometry to map the kelp beds from year to year

In [ ]:
kelp_geo = kelpgdf.copy()

In [ ]:
def prepare_temp(df):

    temp_cols = [
        "Temp",
        "Ave_Temp_Shore_Edge",
        "Ave_Temp_Water_Edge",
        "Temp1_Shore_Edge",
        "Temp2_Shore_Edge",
        "Temp1_Water_Edge",
        "Temp2_Water_Edge",
    ]

    for c in temp_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")


    df["Temp_Final"] = np.where(
        ~df["Temp"].isna(),
        df["Temp"],
        np.where(
            (~df["Ave_Temp_Shore_Edge"].isna()) &
            (~df["Ave_Temp_Water_Edge"].isna()),
            (df["Ave_Temp_Shore_Edge"] + df["Ave_Temp_Water_Edge"]) / 2,
            (
                ((df["Temp1_Shore_Edge"] + df["Temp2_Shore_Edge"]) / 2) +
                ((df["Temp1_Water_Edge"] + df["Temp2_Water_Edge"]) / 2)
            ) / 2
        )
    )

    return df.head()

prepare_temp(kelp_geo)

In [ ]:
kelp_geo['Location'] = kelp_geo['Location'].replace({
    "Ebey's Landing": 'Ebeys Landing'})


In [ ]:
kelp_geo['Survey_Date'] = pd.to_datetime(kelp_geo['Survey_Date'])

In [ ]:
merged_max_acre = pd.merge(max_acres, kelp_geo, on=['site_code','Survey_Date'], how="inner")

In [ ]:
cols_to_keep = [
    "site_code",
    "Location_x",
    "Bed_Name",
    "County",
    "Survey_Date",
    "Survey_Year_x",
    "area_acres",
    "perimeter_miles",
    "Temp_Final",
    "Tidal_Ht_m",
    "Tide_Station",
    "Weather",
    "Observations",
    "geometry",
]

clean_gdf = merged_max_acre[cols_to_keep].copy()

# rename columns
clean_gdf = clean_gdf.rename(
    columns={"Location_x": "Location", "Survey_Year_x": "Survey_Year"}
)

In [ ]:
clean_gdf

In [ ]:
ebey_max = clean_gdf[clean_gdf['Location'] == 'Ebeys Landing']

## Max extent of Ebey from year to year

In [ ]:
import folium
import geopandas as gpd

# survey date as string
gdf = gpd.GeoDataFrame(ebey_max, geometry='geometry', crs='EPSG:4326').copy()
gdf['Survey_Date'] = gdf['Survey_Date'].astype(str)
gdf['Survey_Year'] = gdf['Survey_Year'].astype(str)

# map for Polygon and MultiPolygon
m = folium.Map(location=[48.275, -122.565], tiles='cartodbpositron', zoom_start=14)

folium.GeoJson(
    gdf,
    tooltip=folium.GeoJsonTooltip(fields=['Bed_Name', 'Survey_Year', 'area_acres'])
).add_to(m)

m.fit_bounds(m.get_bounds())
m


# Proposed charts:

1. A map of all of the locations and and ctd popup:

We already have all of the location information we need, and as part of the EDA I have already placed them on a map. We can add a filter that will allow us to see different details about each bed when you hover over it. Ideally, water will be shaded in blue and the land will be grey (so as to imply missing data). However, I think it would make sense to have the kelp bed locations be in green, but that will be hard to see on a blue background.

2. We can go two different ways to visualize the change in area of the beds from year to year. I experimented with plotting them on a map during EDA. If we did this, every year should probably be associated with a different color and be transparent so it does not occulde the years behind it. While I think this might be a compelling graphic for a casual audience, I think it leaves alot to be desired for accuracy of reporting changes in area, which is already a difficult task for a human brain. So the alternate graphic would be such: a bar chart of normalized data that shows the change in area from year to year. Could be a diverging bar chart to really drive home the increase/ decrease from year to year. Position is the most relevant encoding for this type of chart but, if we were to use color, I would use green for growth (inc) and red for decreases. The scale for this type of chart would be percentage change.